# MySQL → Lakehouse Federation → Lakeflow Declarative Pipelines CDC → Medallion Architecture




> Snapshot-diff CDC  --> Each pipeline run re-reads the full table through the foreign catalog, and AUTO CDC FROM SNAPSHOT diffs it against the previous run to infer inserts/updates/deletes ---> Small/medium tables


> Log-based CDC (Lakeflow Connect) ----> A managed connector reads the MySQL binlog directly via a gateway, independent of foreign catalogs --->production, high-volume tables

1. Run this against your MySQL instance MySQL Workbench

`CREATE DATABASE IF NOT EXISTS salesdb;`

`USE salesdb;`

### Dimension: customers (SCD Type 1 target in silver — we only care about current state)


`CREATE TABLE IF NOT EXISTS customers (
    customer_id   BIGINT PRIMARY KEY,
    first_name    VARCHAR(100),
    last_name     VARCHAR(100),
    email         VARCHAR(255),
    city          VARCHAR(100),
    country       VARCHAR(100),
    updated_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP
);`





### Fact: orders (SCD Type 2 target in silver — we want status history)

`CREATE TABLE IF NOT EXISTS orders (
    order_id      BIGINT PRIMARY KEY,
    customer_id   BIGINT,
    order_status  VARCHAR(20),              -- PLACED, PAID, SHIPPED, DELIVERED, CANCELLED
    order_amount  DECIMAL(10,2),
    order_ts      TIMESTAMP,
    updated_at    TIMESTAMP DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    CONSTRAINT fk_orders_customer FOREIGN KEY (customer_id) REFERENCES customers(customer_id)
);`

### initial Load

In [0]:
%sql

INSERT INTO customers (customer_id, first_name, last_name, email, city, country) VALUES
(1, 'Asha',   'Rao',     'asha.rao@example.com',     'Chennai',   'India'),
(2, 'Liam',   'Ng',      'liam.ng@example.com',      'Singapore', 'Singapore'),
(3, 'Maria',  'Silva',   'maria.silva@example.com',  'Lisbon',    'Portugal'),
(4, 'Omar',   'Haddad',  'omar.haddad@example.com',  'Dubai',     'UAE'),
(5, 'Priya',  'Menon',   NULL,                        'Bengaluru', 'India')   
ON DUPLICATE KEY UPDATE first_name = VALUES(first_name);

In [0]:
%sql

INSERT INTO orders (order_id, customer_id, order_status, order_amount, order_ts) VALUES
(1001, 1, 'PLACED',   249.99, NOW()),
(1002, 2, 'PAID',     89.50,  NOW()),
(1003, 3, 'SHIPPED',  512.00, NOW()),
(1004, 4, 'DELIVERED',75.25,  NOW()),
(1005, 1, 'PLACED',   -10.00, NOW())      
ON DUPLICATE KEY UPDATE order_status = VALUES(order_status);

After completing one pipeline run insert into mysql again and try

### INSERT: a brand new order
`
INSERT INTO orders (order_id, customer_id, order_status, order_amount, order_ts)
VALUES (1006, 2, 'PLACED', 129.00, NOW());
`
 
### UPDATE: order progresses through its lifecycle
### in the silver SCD2 table, this closes the PLACED row and opens a PAID row
`
UPDATE orders SET order_status = 'PAID' WHERE order_id = 1001;
`
 
### UPDATE: customer moves city
`
UPDATE customers SET city = 'Coimbatore' WHERE customer_id = 1;
`
 
### FIX: correct the bad seed row so you can show a previously-dropped row
### start passing DQ checks on the next run
`
UPDATE customers SET email = 'priya.menon@example.com' WHERE customer_id = 5;
UPDATE orders SET order_amount = 45.00 WHERE order_id = 1005;
`
 
### DELETE: cancel and remove an order
### AUTO CDC FROM SNAPSHOT detects this automatically on the next pipeline 
`
DELETE FROM orders WHERE order_id = 1004;
`

In [0]:
%sql
select * from retail.gold.customer_ltv